In [16]:
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageFilter, ImageEnhance
import os
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
import random
from concurrent.futures import ThreadPoolExecutor

## chuyển đổi hình ảnh trên tập dữ liệu CIC DDOS 2019

In [17]:

selected_columns = ['tot_fwd_pkts','tot_bwd_pkts','totlen_fwd_pkts','totlen_bwd_pkts',
 'fwd_pkt_len_max','fwd_pkt_len_min','fwd_pkt_len_mean','fwd_pkt_len_std',
 'bwd_pkt_len_max','bwd_pkt_len_min','bwd_pkt_len_mean','bwd_pkt_len_std',
 'flow_byts_s','flow_pkts_s','flow_iat_mean','flow_iat_std','flow_iat_max',
 'flow_iat_min','fwd_iat_tot','fwd_iat_mean','fwd_iat_std','fwd_iat_max',
 'fwd_iat_min','bwd_iat_tot','bwd_iat_mean','bwd_iat_std','bwd_iat_max',
 'bwd_iat_min','fwd_psh_flags','bwd_psh_flags','fwd_urg_flags','bwd_urg_flags',
 'fwd_header_len','bwd_header_len','pkt_len_min','pkt_len_max','pkt_len_mean',
 'pkt_len_std','pkt_len_var','pkt_size_avg','fwd_seg_size_avg','bwd_seg_size_avg',
 'fwd_byts_b_avg','fwd_pkts_b_avg','bwd_byts_b_avg','bwd_pkts_b_avg',
 'subflow_fwd_pkts','subflow_fwd_byts','subflow_bwd_pkts','subflow_bwd_byts',
 'init_fwd_win_byts','init_bwd_win_byts','fwd_act_data_pkts','fwd_seg_size_min',
 'active_mean','active_std','active_max','active_min','idle_mean','idle_std',
 'idle_max','idle_min','cwr_flag_cnt','ece_flag_cnt', 'Label']



selected_columns2 = ['Total Fwd Packets', 'Total Backward Packets',
       'Fwd Packets Length Total', 'Bwd Packets Length Total',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 
       'CWE Flag Count', 'ECE Flag Count',
       'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
       'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 
       'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 
       'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets',
       'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes',
       'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std',
       'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max',
       'Idle Min', 'Label']

len(selected_columns)

65

In [18]:
df2 = pd.read_csv('data/csv/cicddos_2019.csv')
df2 = df2[selected_columns2]

In [19]:
df2[df2["Label"] == "UDP"].sample(10)

,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
30826,2,0,750.0,0.0,375.0,375.0,375.0,0.000000,0.0,0.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
30945,6,0,2088.0,0.0,393.0,321.0,348.0,35.088460,0.0,0.0,...,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
33932,4,0,1438.0,0.0,389.0,330.0,359.5,34.063666,0.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
31271,2,0,766.0,0.0,383.0,383.0,383.0,0.000000,0.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
31618,2,0,750.0,0.0,375.0,375.0,375.0,0.000000,0.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
33237,4,0,1438.0,0.0,389.0,330.0,359.5,34.063666,0.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
32441,4,0,1398.0,0.0,369.0,330.0,349.5,22.516660,0.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
32250,2,0,766.0,0.0,383.0,383.0,383.0,0.000000,0.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
34931,6,0,2088.0,0.0,393.0,321.0,348.0,35.088460,0.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP
32947,2,0,750.0,0.0,375.0,375.0,375.0,0.000000,0.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UDP


In [20]:
df = pd.read_csv('data/csv/udf_loic.csv')
df = df[selected_columns]

In [21]:
df[df["Label"] == "UDP"].sample(10)

,tot_fwd_pkts,tot_bwd_pkts,totlen_fwd_pkts,totlen_bwd_pkts,fwd_pkt_len_max,fwd_pkt_len_min,fwd_pkt_len_mean,fwd_pkt_len_std,bwd_pkt_len_max,bwd_pkt_len_min,...,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,cwr_flag_cnt,ece_flag_cnt,Label
53,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
69,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
124,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
74,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
63,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
93,1,0,8099,0,8099.0,8099.0,8099.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
91,1,0,6401,0,6401.0,6401.0,6401.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
30,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
81,1,0,393,0,393.0,393.0,393.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP
15,1,0,1005,0,1005.0,1005.0,1005.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,UDP


In [22]:

datadir = 'test'
def convert(df_normalized_splited, label, num):
    # Kích thước ảnh ban đầu (5x15) => Mở rộng mỗi điểm thành 3x3 pixel => Ảnh mới (15x45)
    image_size = (8, 8) # kích thước ảnh ban đầu
    upscale_factor = 28 # tỉ lệ tăng kích thước điểm ảnh => size ảnh: (8x28) x (8x28)
    new_image_size = (image_size[0] * upscale_factor, image_size[1] * upscale_factor)

    os.makedirs(f"data/{datadir}/{label}", exist_ok=True)  # Tạo thư mục nếu chưa có

    i = 1
    for row in df_normalized_splited.values:
        # Chuyển đổi dòng thành ma trận ảnh ban đầu (8x8)
        image_array = np.array(row).reshape(image_size)
        image_array = np.nan_to_num(image_array)  # Thay thế giá trị NaN bằng 0

        # Phóng to mỗi pixel np.kron()
        upscale_matrix = np.ones((upscale_factor, upscale_factor))
        enlarged_image_array = np.kron(image_array, upscale_matrix)

        # Chuyển đổi sang ảnh
        image = Image.fromarray((enlarged_image_array * 255).astype(np.uint8))  # Chuyển sang RGB
        image = image.convert("RGB")

        
        ###### thêm nhiễu vào ảnh ######

        # Xoay ảnh ngẫu nhiên (-15° đến 15°)
        rotate_prob = random.random() < 0.8
        if rotate_prob:  
            angle = random.uniform(-90, 90)  
            image = image.rotate(angle)


        # Lật ảnh ngẫu nhiên
        flip_prob = random.random() < 0.8
        if flip_prob:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)  # Lật ngang
            else:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)  # Lật dọc

        # Điều chỉnh độ sáng ngẫu nhiên (từ 70% đến 130%)
        brightness_prob = random.random() < 0.8
        if brightness_prob:
            enhancer = ImageEnhance.Brightness(image)
            factor = random.uniform(0.7, 1.8)  # Thay đổi độ sáng từ 70% đến 130%
            image = enhancer.enhance(factor)


        # Xác suất ngẫu nhiên để thêm hiệu ứng (30% ảnh bị làm mờ, 30% ảnh bị nhiễu)
        blur_prob = random.random() <= 0.5  # 30% xác suất làm mờ
        noise_prob = random.random() <= 0.5  # 30% xác suất thêm nhiễu

        '''
        chỉ làm mờ: 0.3, không mờ 0,7 ==> 0.21 xác xuất chỉ mờ
        chỉ làm nhiễu: 0.3, không nhiễu 0,7 ==> 0.21 xác xuất chỉ nhiễu
        vừa mờ: 0.3, vừa nhiễu 0.3 ==> xác xuất vừa mờ vừa nhiễu: 0.09
        không mờ: 0.7, không nhiễu: 0.7 ==> xác xuất không mờ không nhiễu: 0.49
        '''

        # làm mờ ảnh
        if blur_prob:
            image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(10, 20))) # mức độ mờ từ 1 đến 5

        #làm nhiễu ảnh
        if noise_prob:
            noise = np.random.normal(0, 50, (new_image_size[0], new_image_size[1]))  # Thêm nhiễu Gaussian
            noisy_image_array = np.array(image.convert("L")) + noise  # Chuyển sang grayscale trước khi thêm nhiễu
            noisy_image_array = np.clip(noisy_image_array, 0, 255).astype(np.uint8)  # Giữ giá trị trong khoảng 0-255
            image = Image.fromarray(noisy_image_array).convert("RGB")

        # Lưu ảnh
        image.save(f"data/{datadir}/{label}/{str(i)}.png")

        i += 1
        if i > num:
            break

def setup_to_convert(df_normalized, label):
    os.makedirs(f'data/{datadir}/{label}', exist_ok=True)

    # tổng số lượng ảnh train + valid + test của mỗi nhãn
    n = 200
    convert(df_normalized, label, num=n)

In [24]:
# Nhóm dữ liệu theo cột 'Label'
grouped = df.groupby('Label')

# Tạo dictionary để lưu các DataFrame tương ứng với từng nhãn
dfs = {label: group for label, group in grouped}

def process_label(label, df_label):
    df_drop_label = df_label.drop(columns=['Label'])
    
    df_drop_label.replace([np.inf, -np.inf], np.nan, inplace=True)  # Thay giá trị vô hạn bằng NaN
    df_drop_label.fillna(df_drop_label.median(), inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    data_features = np.log1p(df_drop_label + 1)
    data_standardized = (data_features - data_features.mean()) / data_features.std()
    data_normalized = data_standardized

    setup_to_convert(data_normalized, label)

# Sử dụng ThreadPoolExecutor để chạy đa luồng với tối đa 5 luồng
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(process_label, label, df_label) for label, df_label in dfs.items()]

    # Đợi tất cả các task hoàn thành
    for future in futures:
        future.result()

print("Hoàn thành xử lý đa luồng!")

Hoàn thành xử lý đa luồng!
